In [14]:
from systems import build_random_single_mode, build_kerr_oscillator
import dynamiqs as dq
import jax
import jax.numpy as jnp
import jax

from dynamiqs.steady_state.solvers.steady_state_arnoldi2 import SteadyStateArnoldi, SteadyStateArnoldiResult  
jax.config.update("jax_enable_x64", True)
n=128
#H,Ls = build_random_single_mode(n)
H,Ls = build_kerr_oscillator(n, delta= 2*jnp.pi)
steadystate_arnoldi = dq.SteadyStateArnoldi(tol=1e-6, max_cycles=10, krylov_size=100)
result= steadystate_arnoldi._run(H,Ls,None,None)
rho = result.rho
norminf = jnp.max(jnp.abs(dq.lindbladian(H, Ls,rho).to_jax()))
print(norminf)

4.7500768841465264e-07


In [4]:
jax.config.update("jax_enable_x64", True)
n =30
delta = 35.90
nd =15
deltas = jnp.linspace(-20, 20, nd)*2*jnp.pi
for delta in deltas:
    H, Ls = build_kerr_oscillator(n, delta)
    steadystate_arnoldi = dq.SteadyStateArnoldi(tol=1e-6, max_cycles=0, krylov_size=100,n_refinement=3)
    result= steadystate_arnoldi._run(H,Ls,None,None)
    rho = result.rho
    norminf = jnp.max(jnp.abs(dq.lindbladian(H, Ls,rho).to_jax()))
    print(norminf)

0.00025511123
0.00021789946
0.00011666442
0.000492555
0.00019048572
0.00019407272
0.0003297901
0.040726766
24.833323
12.859559
4.417707
0.096132904
0.00039614952
0.00011282818
0.00023379705


In [ ]:
jax.config.update("jax_enable_x64", True)
n =200
delta = 35.90
nd =15
deltas = jnp.linspace(-20, 20, nd)*2*jnp.pi
for delta in deltas:
    H, Ls = build_kerr_oscillator(n, delta)
    steadystate_arnoldi = dq.SteadyStateArnoldi(tol=1e-6, max_cycles=10, krylov_size=100,n_refinement=3)
    result= steadystate_arnoldi._run(H,Ls,None,None)
    rho = result.rho
    norminf = jnp.max(jnp.abs(dq.lindbladian(H, Ls,rho).to_jax()))
    print(norminf)

In [5]:
from systems import build_random_single_mode, build_kerr_oscillator
import dynamiqs as dq
import jax
import jax.numpy as jnp

import jax
jax.config.update("jax_enable_x64", True)
n=40
H,Ls = build_random_single_mode(n)
steadystate_arnoldi = dq.SteadyStateArnoldi(tol=1e-6, max_cycles=10, krylov_size=100)
result= dq.steadystate(H,Ls, solver=steadystate_arnoldi)
rho = result.rho
norminf = jnp.max(jnp.abs(dq.lindbladian(H, Ls,rho).to_jax()))
print(norminf)

4.582593104091408e-16


In [ ]:
jax.config.update("jax_enable_x64", True)
n =128
delta = 35.90
nd =15
deltas = jnp.linspace(-20, 20, nd)*2*jnp.pi
for delta in deltas:
    H, Ls = build_kerr_oscillator(n, delta)
    steadystate_arnoldi = dq.SteadyStateArnoldi(tol=1e-6, max_cycles=1, krylov_size=300, n_refinement=3)
    result= dq.steadystate(H,Ls, solver=steadystate_arnoldi)
    rho = result.rho
    norminf = jnp.max(jnp.abs(dq.lindbladian(H, Ls,rho).to_jax()))
    print(norminf)

4.1319698179845674e-12
1.195313935836705e-12
1.4802161548256791e-12
8.5158546880848e-12
1.1501193385843783e-11
4.1286158197862154e-12
1.2894502719052727e-11
4.383514588445924e-12
2.429972427895492e-12
2.615347494161314e-12
9.585811411539025e-13
2.3556907773853087e-12


In [4]:
n =40
delta = 0.0
H, Ls = build_kerr_oscillator(n, delta)
#H,Ls = build_random_single_mode(n)
steadystate_gmres = dq.SteadyStateGMRES(tol=1e-6, krylov_size=30, recycling=10,n_refinement=3)
result= dq.steadystate(H,Ls, solver=steadystate_gmres)
rho = result.rho
norminf = jnp.max(jnp.abs(dq.lindbladian(H, Ls,rho).to_jax()))
print(norminf)

66.828049902725


In [3]:
"""
Diagnostic script for IRAM convergence issues.

Run this on a single problematic (delta, system) to understand what's happening.
Adjust `build_kerr_oscillator` import and parameters to match your setup.
"""
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

import dynamiqs as dq

# ── Paste or import your build_kerr_oscillator here ──────────────────
# from your_module import build_kerr_oscillator

# ── Pick a problematic delta ─────────────────────────────────────────
twopi = 2 * jnp.pi
# delta = -35.90 * twopi  # or whichever is worst
# H, Ls = build_kerr_oscillator(n, delta)


# ═══════════════════════════════════════════════════════════════════════
# DIAGNOSTIC 1: Check the spectrum of the Lindbladian directly
# ═══════════════════════════════════════════════════════════════════════
def diagnose_lindbladian_spectrum(H, Ls, n):
    """Build the full Lindbladian superoperator and check its spectrum."""
    dim = n * n
    dtype = H.to_jax().dtype

    # Build full superoperator matrix by applying L to each basis element
    L_matrix = jnp.zeros((dim, dim), dtype=dtype)
    for j in range(dim):
        e_j = jnp.zeros(dim, dtype=dtype).at[j].set(1.0)
        rho_j = e_j.reshape(n, n)
        rho_j_q = dq.asqarray(rho_j, dims=H.dims)
        L_rho_j = dq.lindbladian(H, Ls, rho_j_q).to_jax().ravel()
        L_matrix = L_matrix.at[:, j].set(L_rho_j)

    evals = jnp.linalg.eigvals(L_matrix)

    # Sort by real part (descending)
    idx = jnp.argsort(-jnp.real(evals))
    evals_sorted = evals[idx]

    print("\n╔══════════════════════════════════════════════════════════╗")
    print("║  DIAGNOSTIC 1: Lindbladian spectrum                     ║")
    print("╚══════════════════════════════════════════════════════════╝")
    print(f"  Dimension: {dim}x{dim}")
    print(f"  Eigenvalue closest to 0: {evals_sorted[0]:.6e}")
    print(f"  2nd eigenvalue:          {evals_sorted[1]:.6e}")
    print(f"  Spectral gap |Re(λ₂)|:  {jnp.abs(jnp.real(evals_sorted[1])):.6e}")
    print(f"  3rd eigenvalue:          {evals_sorted[2]:.6e}")
    print(f"\n  Top 10 eigenvalues (by real part):")
    for i in range(min(10, len(evals_sorted))):
        e = evals_sorted[i]
        print(f"    λ_{i}: {jnp.real(e):+.6e} {jnp.imag(e):+.6e}j  "
              f"|λ| = {jnp.abs(e):.6e}")

    return L_matrix, evals_sorted


# ═══════════════════════════════════════════════════════════════════════
# DIAGNOSTIC 2: Check the spectrum of S⁻¹K (the preconditioned operator)
# ═══════════════════════════════════════════════════════════════════════
def diagnose_preconditioned_spectrum(H, Ls, n):
    """Build full matrix of T = S⁻¹K and check its spectrum."""
    from dynamiqs.steady_state.core.arnoldi_iram import arnoldi_iram_lm_jit
    from dynamiqs.steady_state.solvers.steady_state_arnoldi import SteadyStateArnoldi
    # We need to replicate the preconditioner setup from SteadyStateArnoldi

    dims = H.dims
    H_jax = H.to_jax()
    Ls_jax = [L.to_jax() for L in Ls]
    dtype = H_jax.dtype
    dim = n * n

    H_q = H
    Ls_q = dq.stack(Ls)

    from dynamiqs.steady_state.api.utils import (
        from_matrix, to_matrix, to_dm, from_dm, update_preconditioner,
    )
    from dynamiqs.steady_state.preconditionner.lyapunov_solver import LyapunovSolverEig

    identity_vec = from_matrix(jnp.eye(n, dtype=dtype))

    def kraus_vec(x):
        rho = to_dm(x, n=n, dims=dims)
        return from_matrix((Ls_q @ rho @ Ls_q.dag()).sum(0).to_jax())

    LdagL = (Ls_q.dag() @ Ls_q).sum(0).to_jax()
    G = 1j * H_jax + 0.5 * LdagL

    solver = LyapunovSolverEig(G)
    def precond(x):
        return -from_matrix(solver.solve(to_matrix(x, n=n), mu=0.0))

    precond_fn = update_preconditioner(precond, identity_vec, use_rank_1_update=False)

    def precond_kraus(x):
        return precond_fn(kraus_vec(x))

    # Build full matrix of T = precond_kraus
    T_matrix = jnp.zeros((dim, dim), dtype=dtype)
    for j in range(dim):
        e_j = jnp.zeros(dim, dtype=dtype).at[j].set(1.0)
        T_e_j = precond_kraus(e_j)
        T_matrix = T_matrix.at[:, j].set(T_e_j)

    evals = jnp.linalg.eigvals(T_matrix)
    idx = jnp.argsort(-jnp.abs(evals))
    evals_sorted = evals[idx]

    print("\n╔══════════════════════════════════════════════════════════╗")
    print("║  DIAGNOSTIC 2: Spectrum of T = S⁻¹K                    ║")
    print("╚══════════════════════════════════════════════════════════╝")
    print(f"  Top 10 eigenvalues of T (by magnitude):")
    for i in range(min(10, len(evals_sorted))):
        e = evals_sorted[i]
        print(f"    λ_{i}: {jnp.real(e):+.10e} {jnp.imag(e):+.10e}j  "
              f"|λ| = {jnp.abs(e):.10e}")

    gap = jnp.abs(evals_sorted[0]) - jnp.abs(evals_sorted[1])
    ratio = jnp.abs(evals_sorted[1]) / jnp.abs(evals_sorted[0])
    print(f"\n  |λ₁| = {jnp.abs(evals_sorted[0]):.10e}")
    print(f"  |λ₂| = {jnp.abs(evals_sorted[1]):.10e}")
    print(f"  Gap |λ₁|-|λ₂| = {gap:.6e}")
    print(f"  Ratio |λ₂|/|λ₁| = {ratio:.6f}")
    print(f"  (closer to 1 = harder for Arnoldi)")

    return T_matrix, evals_sorted


# ═══════════════════════════════════════════════════════════════════════
# DIAGNOSTIC 3: Run IRAM with verbose convergence history
# ═══════════════════════════════════════════════════════════════════════
def diagnose_iram_convergence(H, Ls, n, krylov_size=30, max_cycles=100, tol=1e-6):
    """Run IRAM and print convergence history."""
    from dynamiqs.steady_state.core.arnoldi_iram import arnoldi_iram_lm_jit
    from dynamiqs.steady_state.api.utils import (
        from_matrix, to_matrix, to_dm, from_dm, update_preconditioner,
    )
    from dynamiqs.steady_state.preconditionner.lyapunov_solver import LyapunovSolverEig

    dims = H.dims
    H_jax = H.to_jax()
    Ls_jax = [L.to_jax() for L in Ls]
    dtype = H_jax.dtype

    H_q = H
    Ls_q = dq.stack(Ls)

    identity_vec = from_matrix(jnp.eye(n, dtype=dtype))

    def kraus_vec(x):
        rho = to_dm(x, n=n, dims=dims)
        return from_matrix((Ls_q @ rho @ Ls_q.dag()).sum(0).to_jax())

    LdagL = (Ls_q.dag() @ Ls_q).sum(0).to_jax()
    G = 1j * H_jax + 0.5 * LdagL
    solver = LyapunovSolverEig(G)
    def precond(x):
        return -from_matrix(solver.solve(to_matrix(x, n=n), mu=0.0))
    precond_fn = update_preconditioner(precond, identity_vec, use_rank_1_update=False)

    def precond_kraus(x):
        return precond_fn(kraus_vec(x))

    def stop_metric(vec):
        rho = to_matrix(vec, n=n)
        rho = 0.5 * (rho + rho.conj().mT)
        tr = jnp.trace(rho)
        tr = jnp.where(jnp.abs(tr) > 0, tr, 1.0)
        rho = rho / tr
        rho_q = dq.asqarray(rho, dims=dims)
        L_rho = dq.lindbladian(H_q, Ls_q, rho_q).to_jax()
        return jnp.max(jnp.abs(L_rho))

    v0 = from_dm(dq.asqarray(jnp.eye(n, dtype=dtype) / n, dims=dims))

    for k_val in [1, 3, 5, 10, min(15, krylov_size - 1)]:
        if k_val >= krylov_size:
            continue
        eigval, eigvec, info = arnoldi_iram_lm_jit(
            precond_kraus, v0,
            m=krylov_size, nev=1, k=k_val,
            max_cycles=max_cycles, tol=tol,
        )

        print(f"\n  ── IRAM  m={krylov_size}, k={k_val}, p={krylov_size - k_val} ──")
        print(f"  Converged: {info.ritz_residual <= tol}")
        print(f"  Cycles: {info.n_cycles}/{max_cycles}")
        print(f"  Eigenvalue: {eigval}")
        print(f"  Ritz residual: {info.ritz_residual:.6e}")
        print(f"  Final metric: {info.metric_hist_cycles[info.n_cycles - 1]:.6e}")

        # Print convergence curve
        n_cyc = int(info.n_cycles)
        hist = info.metric_hist_cycles[:n_cyc]
        ritz = info.ritz_hist_cycles[:n_cyc]
        print(f"  Convergence history (metric | ritz_residual):")
        # Print first 5, last 5
        for i in range(min(5, n_cyc)):
            print(f"    cycle {i:3d}: metric={float(hist[i]):.4e}  ritz={float(ritz[i]):.4e}")
        if n_cyc > 10:
            print(f"    ...")
        for i in range(max(5, n_cyc - 5), n_cyc):
            print(f"    cycle {i:3d}: metric={float(hist[i]):.4e}  ritz={float(ritz[i]):.4e}")


# ═══════════════════════════════════════════════════════════════════════
# RUN ALL DIAGNOSTICS
# ═══════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    # ── FILL IN YOUR SYSTEM HERE ─────────────────────────────────────
    n = 40
    #delta =-20*2*jnp.pi
    delta=-24
    #H, Ls = build_kerr_oscillator(n, delta)
    H, Ls = build_random_single_mode(n)


    L_mat, L_evals = diagnose_lindbladian_spectrum(H, Ls, n)
    T_mat, T_evals = diagnose_preconditioned_spectrum(H, Ls, n)
    diagnose_iram_convergence(H, Ls, n, krylov_size=30)


╔══════════════════════════════════════════════════════════╗
║  DIAGNOSTIC 1: Lindbladian spectrum                     ║
╚══════════════════════════════════════════════════════════╝
  Dimension: 1600x1600
  Eigenvalue closest to 0: 3.902083e-15+2.964132e-14j
  2nd eigenvalue:          -2.736015e-01-1.042500e-15j
  Spectral gap |Re(λ₂)|:  2.736015e-01
  3rd eigenvalue:          -2.857678e-01+3.755558e-03j

  Top 10 eigenvalues (by real part):
    λ_0: +3.902083e-15 +2.964132e-14j  |λ| = 2.989706e-14
    λ_1: -2.736015e-01 -1.042500e-15j  |λ| = 2.736015e-01
    λ_2: -2.857678e-01 +3.755558e-03j  |λ| = 2.857925e-01
    λ_3: -2.857678e-01 -3.755558e-03j  |λ| = 2.857925e-01
    λ_4: -3.007697e-01 +3.242038e-16j  |λ| = 3.007697e-01
    λ_5: -3.198131e-01 -9.471765e-03j  |λ| = 3.199533e-01
    λ_6: -3.198131e-01 +9.471765e-03j  |λ| = 3.199533e-01
    λ_7: -3.215193e-01 +2.484248e-15j  |λ| = 3.215193e-01
    λ_8: -3.235038e-01 -3.934228e-15j  |λ| = 3.235038e-01
    λ_9: -3.245785e-01 +7.63312

In [7]:
# Test 3: bypass custom_linear_solve entirely
from dynamiqs.steady_state.core.arnoldi_iram import arnoldi_iram_lm_jit
from dynamiqs.steady_state.api.utils import (
    from_matrix, to_matrix, to_dm, from_dm, update_preconditioner,
)
from dynamiqs.steady_state.preconditionner.lyapunov_solver import LyapunovSolverEig

n = 40
delta = 35.90
H, Ls = build_kerr_oscillator(n, delta)
#H, Ls = build_random_single_mode(n)
dims = H.dims
H_jax = H.to_jax()
dtype = H_jax.dtype
Ls_q = dq.stack(Ls)

identity_vec = from_matrix(jnp.eye(n, dtype=dtype))

def kraus_vec(x):
    rho = to_dm(x, n=n, dims=dims)
    return from_matrix((Ls_q @ rho @ Ls_q.dag()).sum(0).to_jax())

LdagL = (Ls_q.dag() @ Ls_q).sum(0).to_jax()
G = 1j * H_jax + 0.5 * LdagL
solver_lyap = LyapunovSolverEig(G)
def precond(x):
    return -from_matrix(solver_lyap.solve(to_matrix(x, n=n), mu=0.0))

def precond_kraus(x):
    return precond(kraus_vec(x))

def stop_metric(vec):
    rho = to_matrix(vec, n=n)
    rho = 0.5 * (rho + rho.conj().mT)
    tr = jnp.trace(rho)
    tr = jnp.where(jnp.abs(tr) > 0, tr, 1.0)
    rho = rho / tr
    rho_q = dq.asqarray(rho, dims=dims)
    return jnp.max(jnp.abs(dq.lindbladian(H, Ls, rho_q).to_jax()))

v0 = from_dm(dq.asqarray(jnp.eye(n, dtype=dtype) / n, dims=dims))

eigval, eigvec, info = arnoldi_iram_lm_jit(
    precond_kraus, v0,
    m=30, nev=1, k=10,
    max_cycles=100, tol=1e-6,
    stop_fn=stop_metric,
)

# Direct hermitise + normalise (NO custom_linear_solve)
rho_direct = to_matrix(eigvec, n=n)
rho_direct = 0.5 * (rho_direct + rho_direct.conj().mT)
rho_direct = rho_direct / jnp.trace(rho_direct)
norm_direct = jnp.max(jnp.abs(dq.lindbladian(H, Ls, dq.asqarray(rho_direct, dims=dims)).to_jax()))

print(f"IRAM direct (no custom_linear_solve): {norm_direct:.6e}")
print(f"IRAM info metric:                     {info.metric_hist_cycles[info.n_cycles-1]:.6e}")
print(f"IRAM ritz residual:                   {info.ritz_residual:.6e}")
print(f"IRAM cycles:                          {info.n_cycles}")

eigval, eigvec, info = arnoldi_iram_lm_jit(
    precond_kraus, v0,
    m=30, nev=1, k=10,
    max_cycles=100, tol=1e-6,
)

# Direct hermitise + normalise (NO custom_linear_solve)
rho_direct = to_matrix(eigvec, n=n)
rho_direct = 0.5 * (rho_direct + rho_direct.conj().mT)
rho_direct = rho_direct / jnp.trace(rho_direct)
norm_direct = jnp.max(jnp.abs(dq.lindbladian(H, Ls, dq.asqarray(rho_direct, dims=dims)).to_jax()))

print(f"IRAM direct no stop fn (no custom_linear_solve): {norm_direct:.6e}")
print(f"IRAM info metric:                     {info.metric_hist_cycles[info.n_cycles-1]:.6e}")
print(f"IRAM ritz residual:                   {info.ritz_residual:.6e}")
print(f"IRAM cycles:                          {info.n_cycles}")

IRAM direct (no custom_linear_solve): 9.564157e-03
IRAM info metric:                     6.649345e-02
IRAM ritz residual:                   0.000000e+00
IRAM cycles:                          100
IRAM direct no stop fn (no custom_linear_solve): 2.562653e-01
IRAM info metric:                     7.711959e-07
IRAM ritz residual:                   7.711959e-07
IRAM cycles:                          6


In [6]:
# Pour le Kerr oscillator problématique
n = 40
delta = 35.90
H, Ls = build_kerr_oscillator(n, delta)
# ... (même setup que ton test)

eigval, eigvec, info = arnoldi_iram_lm_jit(
    precond_kraus, v0,
    m=30, nev=1, k=10,
    max_cycles=100, tol=1e-8,
)

rho_raw = to_matrix(eigvec, n=n)

# Diagnostic de la phase
print(f"Eigenvalue: {eigval}")
print(f"Trace of raw rho: {jnp.trace(rho_raw)}")
print(f"||rho - rho†|| / ||rho||: {jnp.linalg.norm(rho_raw - rho_raw.conj().T) / jnp.linalg.norm(rho_raw):.6e}")

# Fix: rotate the eigenvector so that trace is real and positive
tr = jnp.trace(rho_raw)
phase = tr / jnp.abs(tr)  # unit complex number
rho_fixed = rho_raw / phase  # now trace is real positive

print(f"\nAfter phase fix:")
print(f"Trace: {jnp.trace(rho_fixed)}")
print(f"||rho - rho†|| / ||rho||: {jnp.linalg.norm(rho_fixed - rho_fixed.conj().T) / jnp.linalg.norm(rho_fixed):.6e}")

rho_fixed = 0.5 * (rho_fixed + rho_fixed.conj().T)
rho_fixed = rho_fixed / jnp.trace(rho_fixed)
norm_fixed = jnp.max(jnp.abs(dq.lindbladian(H, Ls, dq.asqarray(rho_fixed, dims=dims)).to_jax()))
print(f"Physical metric after phase fix: {norm_fixed:.6e}")

Eigenvalue: (-1.0000000000000038+2.8796409701215e-16j)
Trace of raw rho: (6.272172435844777-1.4869205397504267e-14j)
||rho - rho†|| / ||rho||: 4.887366e-15

After phase fix:
Trace: (6.272172435844777-6.890206969039775e-30j)
||rho - rho†|| / ||rho||: 1.191728e-15
Physical metric after phase fix: 8.439202e+01


In [21]:
# Test: IRAM sans stop_fn, puis vérification physique après
eigval, eigvec, info = arnoldi_iram_lm_jit(
    precond_kraus, v0,
    m=30, nev=1, k=10,
    max_cycles=100, tol=1e-8,  # tight Ritz tolerance
    # PAS de stop_fn
)

rho_direct = to_matrix(eigvec, n=n)
rho_direct = 0.5 * (rho_direct + rho_direct.conj().mT)
rho_direct = rho_direct / jnp.trace(rho_direct)
norm_direct = jnp.max(jnp.abs(dq.lindbladian(H, Ls, dq.asqarray(rho_direct, dims=dims)).to_jax()))

print(f"Eigenvalue: {eigval}")
print(f"Ritz residual: {info.ritz_residual:.6e}")
print(f"Cycles: {info.n_cycles}")
print(f"Physical metric: {norm_direct:.6e}")

Eigenvalue: (-0.9995486140435998+0.00041852735157721247j)
Ritz residual: 2.366650e-09
Cycles: 8
Physical metric: 1.295333e-02


In [19]:
"""Benchmark: Jacobian of steady-state loss function.

Compares dense vs GMRES+precond at various krylov sizes, for forward + Jacobian.
Also compares vmap vs lax.map batching.

No catographer dependency. N=30 to keep runtime reasonable.
"""

import gc
import time

import dynamiqs as dq
import jax
import jax.numpy as jnp

dq.set_matmul_precision("highest")
dq.set_precision("double")

N = 40
a = dq.destroy(N)
n_hat_jax = dq.number(N).to_jax()
n = N
twopi = 2 * jnp.pi

# Precomputed operator matrices
a_jax = a.to_jax()
adag_jax = a.dag().to_jax()
adag2a2 = (a.dag() @ a.dag() @ a @ a).to_jax()
adaga = (a.dag() @ a).to_jax()
I_vec = jnp.eye(n, dtype=jnp.complex128).flatten(order="F")

n_detunings = 15
delta_vals = jnp.linspace(-20, 20, n_detunings) * twopi

# Fit parameters: [kappa, kerr, eps]
params_true = jnp.array([14.0 * twopi, -1.0 * twopi, 16.0])


def build_H_and_Ls(params, delta):
    kap, kerr, ep = params
    H = (
        -kerr / 2 * adag2a2
        - delta * adaga
        + 1j * jnp.sqrt(kap) * ep * a_jax
        - 1j * jnp.sqrt(kap) * ep * adag_jax
    )
    L = jnp.sqrt(kap) * a_jax
    return dq.asqarray(H), [dq.asqarray(L)]


# Dense: build superoperator, direct solve
def dense_single(params, delta):
    H_q, Ls_q = build_H_and_Ls(params, delta)
    L_sup = dq.slindbladian(H_q, Ls_q).to_jax()
    L_def = L_sup + jnp.outer(I_vec, I_vec)
    x = jnp.linalg.solve(L_def, I_vec)
    rho = x.reshape((n, n), order="F")
    rho = (rho + rho.conj().T) / 2
    rho /= jnp.trace(rho)
    return jnp.trace(rho @ n_hat_jax).real


# GMRES with Lyapunov preconditioner (dynamiqs)
def gmres_single(params, delta, ks=64):
    H_q, Ls_q = build_H_and_Ls(params, delta)
    solver = dq.SteadyStateArnoldi(tol=1e-6, max_cycles=10, krylov_size=ks)
    result = dq.steadystate(H_q, Ls_q, solver=solver)
    return jnp.trace(result.rho.to_jax() @ n_hat_jax).real


# ── Compute dense reference ──────────────────────────────────────
print(f"Kerr oscillator: N={N}, {n_detunings} detunings, 3 fit params")
print("Jacobian: jacfwd of <n>(params) w.r.t. [κ, K, ε]")
print("Computing dense reference...")
ref_fwd_fn = jax.jit(lambda p: jax.lax.map(lambda d: dense_single(p, d), delta_vals))
n_ref = ref_fwd_fn(params_true).block_until_ready()
ref_jac_fn = jax.jit(
    jax.jacfwd(lambda p: jax.lax.map(lambda d: dense_single(p, d), delta_vals))
)
jac_ref = ref_jac_fn(params_true).block_until_ready()
print("Done.\n")


def run_benchmark(label, single_fn, krylov_sizes):
    print(f"\n{'=' * 95}")
    print(f"{label}")
    print(f"{'=' * 95}")
    print(
        f"  {'ks':>4}  {'mode':>8}  {'fwd(ms)':>8}  {'jac(ms)':>8}  "
        f"{'total(ms)':>10}  {'fwd_err':>10}  {'jac_err':>10}"
    )
    print(f"  {'-' * 82}")

    for ks in krylov_sizes:
        if ks is None:
            solve_fn = single_fn
        else:
            solve_fn = lambda p, d, _ks=ks: single_fn(p, d, _ks)

        for mode in ["vmap", "lax.map"]:
            jax.clear_caches()
            gc.collect()

            if mode == "vmap":
                batched = lambda p: jax.vmap(solve_fn, in_axes=(None, 0))(p, delta_vals)
            else:
                batched = lambda p: jax.lax.map(lambda d: solve_fn(p, d), delta_vals)

            # Forward
            fn_fwd = jax.jit(batched)
            _ = fn_fwd(params_true).block_until_ready()
            ts = []
            for _ in range(3):
                t0 = time.time()
                vals = fn_fwd(params_true).block_until_ready()
                ts.append(time.time() - t0)
            t_fwd = min(ts) * 1000
            fwd_err = float(jnp.max(jnp.abs(vals - n_ref)))

            # Jacobian
            fn_jac = jax.jit(jax.jacfwd(batched))
            _ = fn_jac(params_true).block_until_ready()
            ts = []
            for _ in range(3):
                t0 = time.time()
                jac = fn_jac(params_true).block_until_ready()
                ts.append(time.time() - t0)
            t_jac = min(ts) * 1000
            jac_err = float(jnp.max(jnp.abs(jac - jac_ref)))
            has_nan = bool(jnp.any(jnp.isnan(jac)))

            ks_str = f"{ks}" if ks is not None else "  —"
            nan_tag = " NaN!" if has_nan else ""
            wrong_tag = " ←WRONG" if fwd_err > 0.1 else ""
            print(
                f"  {ks_str:>4}  {mode:>8}  {t_fwd:8.0f}  {t_jac:8.0f}  "
                f"{t_fwd + t_jac:10.0f}  {fwd_err:10.2e}{wrong_tag}  "
                f"{jac_err:10.2e}{nan_tag}"
            )


# ── Run benchmarks ───────────────────────────────────────────────
run_benchmark("DENSE SOLVER", dense_single, [None])

run_benchmark(
    "GMRES",
    gmres_single,
    [32, 48, 64, 96],
)

Kerr oscillator: N=40, 15 detunings, 3 fit params
Jacobian: jacfwd of <n>(params) w.r.t. [κ, K, ε]
Computing dense reference...
Done.


DENSE SOLVER
    ks      mode   fwd(ms)   jac(ms)   total(ms)     fwd_err     jac_err
  ----------------------------------------------------------------------------------


KeyboardInterrupt: 

In [1]:
"""
Benchmark: Jacobian of steady-state loss function (sequential solves).

- Computes dense reference (batched) once (printed as 1 summary line).
- Then runs GMRES solvers per detuning with detailed per-line output:
    delta | fwd | fwd_err | jacfwd | jac_err | max|L(rho)|

No cartographer dependency. N=40.
"""

import gc
import time

import dynamiqs as dq
import jax
import jax.numpy as jnp

# -----------------------------------------------------------------------------
# Global settings
# -----------------------------------------------------------------------------
dq.set_matmul_precision("highest")
dq.set_precision("double")

# -----------------------------------------------------------------------------
# Problem setup
# -----------------------------------------------------------------------------
N = 40
n = N
twopi = 2 * jnp.pi

a = dq.destroy(N)
n_hat_jax = dq.number(N).to_jax()

a_jax = a.to_jax()
adag_jax = a.dag().to_jax()
adag2a2 = (a.dag() @ a.dag() @ a @ a).to_jax()
adaga = (a.dag() @ a).to_jax()

I_vec = jnp.eye(n, dtype=jnp.complex128).flatten(order="F")

n_detunings = 15
delta_vals = jnp.linspace(-20, 20, n_detunings) * twopi

# Fit parameters: [kappa, kerr, eps]
params_true = jnp.array([14.0 * twopi, -1.0 * twopi, 16.0])


def build_H_and_Ls(params, delta):
    kap, kerr, ep = params
    H = (
        -kerr / 2 * adag2a2
        - delta * adaga
        + 1j * jnp.sqrt(kap) * ep * a_jax
        - 1j * jnp.sqrt(kap) * ep * adag_jax
    )
    L = jnp.sqrt(kap) * a_jax
    return dq.asqarray(H), [dq.asqarray(L)]


# -----------------------------------------------------------------------------
# Solvers
# -----------------------------------------------------------------------------
def dense_single_with_rho(params, delta):
    H_q, Ls_q = build_H_and_Ls(params, delta)

    L_sup = dq.slindbladian(H_q, Ls_q).to_jax()
    L_def = L_sup + jnp.outer(I_vec, I_vec)

    x = jnp.linalg.solve(L_def, I_vec)
    rho = x.reshape((n, n), order="F")

    rho = (rho + rho.conj().T) / 2
    rho /= jnp.trace(rho)

    nval = jnp.trace(rho @ n_hat_jax).real
    return nval, rho, H_q, Ls_q


def gmres_single_with_rho(params, delta, ks=64):
    H_q, Ls_q = build_H_and_Ls(params, delta)
    solver = dq.SteadyStateArnoldi(tol=1e-6, max_cycles=10, krylov_size=ks)
    result = dq.steadystate(H_q, Ls_q, solver=solver)

    rho = result.rho.to_jax()
    nval = jnp.trace(rho @ n_hat_jax).real
    return nval, rho, H_q, Ls_q


# -----------------------------------------------------------------------------
# Dense reference (batched) — computed once, printed as 1 summary line
# -----------------------------------------------------------------------------
print(f"Kerr oscillator: N={N}, {n_detunings} detunings, 3 fit params")
print("Jacobian: jacfwd of <n>(params) w.r.t. [κ, K, ε]")
print("Computing dense reference (batched)...\n")

ref_fwd_fn = jax.jit(
    lambda p: jax.lax.map(lambda d: dense_single_with_rho(p, d)[0], delta_vals)
)
n_ref = ref_fwd_fn(params_true).block_until_ready()

ref_jac_fn = jax.jit(
    jax.jacfwd(lambda p: jax.lax.map(lambda d: dense_single_with_rho(p, d)[0], delta_vals))
)
jac_ref = ref_jac_fn(params_true).block_until_ready()

# Compute max Lindbladian norm across all detunings for the dense solver
dense_lind_norms = []
fwd_tuple_dense = jax.jit(lambda p, d: dense_single_with_rho(p, d))
for d in list(delta_vals):
    _, rho, H_q, Ls_q = fwd_tuple_dense(params_true, d)
    Lrho = dq.lindbladian(H_q, Ls_q, dq.asqarray(rho)).to_jax()
    dense_lind_norms.append(float(jnp.max(jnp.abs(Lrho))))

print(f"{'='*80}")
print(f"DENSE REFERENCE (summary)")
print(f"{'='*80}")
print(
    f"  <n> range: [{float(jnp.min(n_ref)):.6e}, {float(jnp.max(n_ref)):.6e}]  "
    f"max|L(rho)|: {max(dense_lind_norms):.2e}"
)
print(f"{'='*80}")


# -----------------------------------------------------------------------------
# GMRES: detailed per-detuning output
# -----------------------------------------------------------------------------
def print_gmres_per_system(ks):
    label = f"GMRES(ks={ks})"
    print(f"\n{'='*150}")
    print(f"{label} — per detuning: delta | fwd | fwd_err | jacfwd | jac_err | max|L(rho)|")
    print(f"{'='*150}")
    print(
        f"{'delta(rad/s)':>14}  {'fwd(<n>)':>12}  {'fwd_err':>10}  "
        f"{'jacfwd[dκ,dK,dε]':>44}  {'jac_maxerr':>10}  {'max|L(rho)|':>14}"
    )
    print(f"{'-'*150}")

    solve = lambda p, d: gmres_single_with_rho(p, d, ks)

    fwd_tuple = jax.jit(lambda p, d: solve(p, d))
    jac_one = jax.jit(jax.jacfwd(lambda p, d: solve(p, d)[0]))

    # warmups
    n0, rho0, _, _ = fwd_tuple(params_true, delta_vals[0])
    n0.block_until_ready()
    _ = jac_one(params_true, delta_vals[0]).block_until_ready()

    for i, d in enumerate(list(delta_vals)):
        nval, rho, H_q, Ls_q = fwd_tuple(params_true, d)
        grad = jac_one(params_true, d)

        nval.block_until_ready()
        grad.block_until_ready()

        fwd_err = float(jnp.abs(nval - n_ref[i]))
        jac_err = float(jnp.max(jnp.abs(grad - jac_ref[i])))

        Lrho = dq.lindbladian(H_q, Ls_q, dq.asqarray(rho)).to_jax()
        lind_norm = jnp.max(jnp.abs(Lrho))
        lind_norm.block_until_ready()

        print(
            f"{float(d):14.6e}  "
            f"{float(nval):12.6e}  "
            f"{fwd_err:10.2e}  "
            f"[{float(grad[0]): .3e}, {float(grad[1]): .3e}, {float(grad[2]): .3e}]  "
            f"{jac_err:10.2e}  "
            f"{float(lind_norm):14.6e}"
        )


# -----------------------------------------------------------------------------
# Sequential benchmark runner (timing + aggregate errors)
# -----------------------------------------------------------------------------
def run_benchmark_sequential(label, single_with_rho_fn, krylov_sizes):
    print(f"\n{'=' * 95}")
    print(f"{label} (sequential per-detuning)")
    print(f"{'=' * 95}")
    print(
        f"  {'ks':>4}  {'fwd_total(ms)':>13}  {'jac_total(ms)':>13}  "
        f"{'total(ms)':>10}  {'fwd_err':>10}  {'jac_err':>10}"
    )
    print(f"  {'-' * 82}")

    for ks in krylov_sizes:
        if ks is None:
            solve = lambda p, d: single_with_rho_fn(p, d)
        else:
            solve = lambda p, d: single_with_rho_fn(p, d, ks)

        fwd_one = jax.jit(lambda p, d: solve(p, d)[0])
        jac_one = jax.jit(jax.jacfwd(lambda p, d: solve(p, d)[0]))

        jax.clear_caches()
        gc.collect()

        # ---- Forward timing ----
        _ = fwd_one(params_true, delta_vals[0]).block_until_ready()
        t0 = time.time()
        vals = []
        for d in list(delta_vals):
            vals.append(fwd_one(params_true, d).block_until_ready())
        t_fwd = (time.time() - t0) * 1000.0
        vals = jnp.stack(vals)
        fwd_err = float(jnp.max(jnp.abs(vals - n_ref)))

        # ---- Jacobian timing ----
        _ = jac_one(params_true, delta_vals[0]).block_until_ready()
        t0 = time.time()
        jacs = []
        for d in list(delta_vals):
            jacs.append(jac_one(params_true, d).block_until_ready())
        t_jac = (time.time() - t0) * 1000.0
        jac = jnp.stack(jacs)

        jac_err = float(jnp.max(jnp.abs(jac - jac_ref)))
        has_nan = bool(jnp.any(jnp.isnan(jac)))

        ks_str = f"{ks}" if ks is not None else "  —"
        nan_tag = " NaN!" if has_nan else ""
        wrong_tag = " ←WRONG" if fwd_err > 0.1 else ""
        print(
            f"  {ks_str:>4}  {t_fwd:13.0f}  {t_jac:13.0f}  "
            f"{t_fwd + t_jac:10.0f}  {fwd_err:10.2e}{wrong_tag}  "
            f"{jac_err:10.2e}{nan_tag}"
        )


# -----------------------------------------------------------------------------
# Run
# -----------------------------------------------------------------------------

for ks in [32, 48, 64, 96]:
    print_gmres_per_system(ks)

Kerr oscillator: N=40, 15 detunings, 3 fit params
Jacobian: jacfwd of <n>(params) w.r.t. [κ, K, ε]
Computing dense reference (batched)...

DENSE REFERENCE (summary)
  <n> range: [1.152176e+00, 1.038948e+01]  max|L(rho)|: 1.33e-13

GMRES(ks=32) — per detuning: delta | fwd | fwd_err | jacfwd | jac_err | max|L(rho)|
  delta(rad/s)      fwd(<n>)     fwd_err                              jacfwd[dκ,dK,dε]  jac_maxerr     max|L(rho)|
------------------------------------------------------------------------------------------------------------------------------------------------------
 -1.256637e+02  1.152175e+00    1.46e-07  [ 9.578e-03,  1.560e-02,  1.314e-01]    3.70e-08    4.430215e-06
 -1.077117e+02  1.450849e+00    1.07e-07  [ 1.091e-02,  2.641e-02,  1.600e-01]    2.57e-07    2.115247e-06
 -8.975979e+01  1.853911e+00    7.47e-08  [ 1.204e-02,  4.550e-02,  1.949e-01]    2.07e-07    1.118975e-06
 -7.180783e+01  2.395747e+00    1.79e-07  [ 1.252e-02,  7.866e-02,  2.357e-01]    5.33e-06    2.87

KeyboardInterrupt: 